# AGENT11 (en construction)

## Généralisation des schémas spatio séquentiels

D'une manière générale, il faut pouvoir représenter des motifs spatio sequentiels d'événements. 

* Les `CompositeInteraction` stockent leurs sous-interaction dans un dictionnaire dont la clé est leur `PositionIndex` spatio-temporell.
* Les `CompositeInteraction` dont toutes les sous-interaction sont simultanées sont des _Evénements composites_. Elles n'ont pas de post_interaction.
    * Tant que l'agent ne pourra prendre qu'une décision à la fois, elles devront toutes avoir la même décision

# L'environnement 

Nous repartons de l'environnement de Agent10. 

Nous ajoutons une classe `Pose` pour faciliter le calcul des déplacements de l'agent dans l'environnement et dans le simulateur.

In [1]:
import numpy as np

# save_dir = "sav"

LEFT = 0
DOWN = 1
RIGHT = 2
UP = 3

FORWARD = 2  #0
FEEL = 1
# FEEL_LEFT = 2
# FEEL_RIGHT = 3
TURN_LEFT = 4
TURN_RIGHT = 5

# The size of the simulateor should be even to place the agent at the center
ENV_HIGHT = 5
ENV_WIDTH = 5
SIM_HIGHT = 5
SIM_WIDTH = 5

ENV_SIZE = np.array([ENV_HIGHT, ENV_WIDTH])
DIRECTIONS = np.array([
            [0, -1],  # Left
            [1, 0],   # Down
            [0, 1],   # Right
            [-1, 0]   # Up
            ])

In [120]:
class Pose:
    """The agent's pose defines the agent's coordinates and direction"""
    def __init__(self, position, direction):
        """Intialize the pose from a tuple (line, column) and a direction"""
        self.position = np.array(position)  # Must be cloned
        self.direction = direction
    def front(self):
        """Return the position in front of the agent as a tuple of int"""
        return tuple(np.mod(self.position + DIRECTIONS[self.direction], ENV_SIZE).tolist())
    def left(self):
        """Return the position on the left of the agent as a tuble of int"""
        return tuple(np.mod(self.position + DIRECTIONS[(self.direction + 1) % 4], ENV_SIZE).tolist())
    def right(self):
        """Return the position on the right as a tuple as a tuble of int"""
        return tuple(np.mod(self.position + DIRECTIONS[self.direction - 1], ENV_SIZE).tolist())
    def forward(self):
        """Move the agent one cell forward"""
        self.position[:] = np.mod(self.position + DIRECTIONS[self.direction], ENV_SIZE)
    def turn_left(self):
        """Turn the agent to the left (counterclockwise)"""
        self.direction = {LEFT: DOWN, DOWN: RIGHT, RIGHT: UP, UP: LEFT}[self.direction]
    def turn_right(self):
        """Turn the agent to the right (clockwise)"""
        self.direction = {LEFT: UP, DOWN: LEFT, RIGHT: DOWN, UP: RIGHT}[self.direction]
    def line(self):
        """Return the line for display (first coordinate of the position)"""
        return self.position[0]
    def column(self):
        """Return the column for display (second coordinate of the position)"""
        return self.position[1]
    def marker(self):
        """Return the marker for display"""
        return {LEFT: '<', DOWN: 'v', RIGHT: '>', UP: '^'}[self.direction]
    def clone(self):
        """Return a clone of the pose"""
        return Pose(self.position, self.direction)
    def __repr__(self):
        """Return a textual representation for debug"""
        return f"({self.line()},{self.column()}){self.marker()}"
    def __eq__(self, other):
        """Pose are equal if their positions and direction are equal"""
        return np.array_equal(self.position, other.position) and self.direction == other.direction

In [121]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from ipywidgets import Button, HBox,VBox, Output
from IPython.display import display

EMPTY = 0
WALL = 1
FEELING_EMPTY = 2
FEELING_WALL = 3
BUMPING = 4
UNKNOWN = 5
PREDICTION_ERROR = 6

colors = ["#b0b0b0", '#b0b0b0', '#ffffff', '#535865', "#F93943"]  # Hidden environment
colors = ["#D6D6D6", '#5C946E', '#FAE2DB', '#535865', "#F93943", 
          "#EEEEEE", "#85A48F", '#FAE2DB', '#535865', "#F93943", '#BAC9E1', "#f62dae"]  # Simulator contains additional UNKNOWN et ERROR
agent_color = "#1976D2"
simulator_agent_color = "#aaaaaa"
prediction_error_color = "#f62dae"
agent_size = 200


class SmallLoop():
    def __init__(self, position, direction, grid):
        self.environment_grid = np.array(grid)
        self.display_grid = np.full((ENV_HIGHT, ENV_WIDTH + SIM_WIDTH), WALL, dtype=int)
        self.display_grid[0:self.environment_grid.shape[0], 0:self.environment_grid.shape[1]] = self.environment_grid
        self.pose = Pose(position, direction)
        self.cmap = ListedColormap(colors)
        self.norm = BoundaryNorm(np.arange(-0.5, len(colors) + 0.5, 1.0), self.cmap.N)
        self.marker_size = agent_size
        self.marker_color = agent_color

    def outcome(self, action):
        """Update the grid. Return the outcome of the action."""
        result = [EMPTY]
        self.display_grid[0:self.environment_grid.shape[0], 0:self.environment_grid.shape[1]] = self.environment_grid

        if action == FORWARD:  
            front = self.pose.front()
            if self.environment_grid[front] == EMPTY:
                self.pose.forward()
            else:
                result = [WALL]
                self.display_grid[front] = BUMPING
        elif action == TURN_RIGHT:
            self.pose.turn_right()
        elif action == TURN_LEFT:
            self.pose.turn_left()
        elif action == FEEL:
            # Decision FEEL_FRONT now yields 3 outcomes for front, left, and Right
            result = []
            for feeling_position in [self.pose.front(), self.pose.left(), self.pose.right()]:
                if self.environment_grid[feeling_position] == EMPTY:
                    self.display_grid[feeling_position] = FEELING_EMPTY
                    result.append(EMPTY)
                else:
                    result.append(WALL)
                    self.display_grid[feeling_position] = FEELING_WALL

        # print(f"Line: {self.position[0]}, Column: {self.position[1]}, direction: {self.direction}")
        return result  
    
    def display(self, simulator=None):
        """Display the grid in the notebook"""
        out.clear_output(wait=True)
        with out:
            fig, ax = plt.subplots()
            if simulator is not None:
                plt.scatter(simulator.pose.column() + ENV_WIDTH, simulator.pose.line(), s=simulator.marker_size, marker=simulator.pose.marker(), 
                            c=simulator.marker_color)
                self.display_grid[0:SIM_HIGHT, ENV_WIDTH:(ENV_WIDTH + SIM_WIDTH + 1)] = simulator.display_grid[0:SIM_HIGHT, 0:SIM_WIDTH] + 5
            ax.imshow(self.display_grid, cmap=self.cmap, norm=self.norm)
            plt.scatter(self.pose.column(), self.pose.line(), s=self.marker_size, marker=self.pose.marker(), c=self.marker_color)
            ax.text(3.5, 0, f"{step:>3}", fontsize=12, color='White')
            plt.show()
            plt.pause(0.01)
    
    def save(self, step, img_nb, simulator):
        """Save the display as a PNG file"""
        fig, ax = plt.subplots()
        ax.set_xticks([])
        ax.set_yticks([])
        ax.axis('off')
        ax.imshow(self.display_grid, cmap=self.cmap, norm=self.norm)
        plt.scatter(self.pose.column(), self.pose.line(), s=self.marker_size, marker=self.pose.marker(), c=self.marker_color)
        plt.scatter(simulator.pose.column() + ENV_WIDTH, simulator.pose.line(), s=simulator.marker_size, marker=simulator.pose.marker(), 
                    c=simulator.marker_color)
        ax.text(3.5, 0, f"{step:>4}", fontsize=10, color='White')
        plt.savefig(f"{save_dir}/{img_nb:04}.png", bbox_inches='tight', pad_inches=0, transparent=True)
        plt.close(fig)
    
    def clear(self, clear):
        """Clear the grid display"""
        if clear:
            self.display_grid[0:6, 0:6] = self.environment_grid
       

# Les classes Interaction

Les Interactions de l'agent geith/Agent10 sont modifiées pour représenter des schémas spatio-temporels:
* La classe `LocalizedInteract` est suppriméd
* La nouvelle classe `Coordinate` est créée pour spécifier les coordonnées spatio-temporelless d'un EIV.
    * Une `Coordinate` sert de key pour les EIVs consitituant une `CompositeInteraction`.
    * Deux `CompositeInteraction` sont égales si elles ont le même dictionnaire de sous-interactions
    * Leur `Key` contient à la fois la hierarchie temporelle et les EIVs synchrones. 

In [122]:
PRE = 0  # Temporal coordinate of the pre interaction
POST = 1 # Temporal coordinate of the post interaction

class CompositeInteraction:
    """A composite interaction is a tuple (pre_interaction, post_interaction) and a weight"""
    def __init__(self, interactions):
        """Construct the composite interaction from a dictionary of localized interaction"""
        self._interactions = interactions
        assert all(isinstance(k, PositionIndex) for k,p in self._interactions.items()), "All keys must be Coordinate"
        self.weight = 1
        self._step = 1

    def simultaneous_interactions(self, temporal):
        """Return the dictionary of interactions that match the temporal coordinate"""
        return {k: i for k, i in self._interactions.items() if k.get_temporal() == temporal}

    def get_pre_interaction(self):
        # Prepare for composite interaction that may contain several interactions at different position at time 0
        pre_interactions = [p for c, p in self._interactions.items() if c.get_temporal()==PRE]
        return pre_interactions[0]

    def get_post_interaction(self):
        # Prepare for composite interactions that may contain several interactions at different positions at time 1
        post_interactions = [p for c, p in self._interactions.items() if c.get_temporal()==POST]
        if len(post_interactions) == 0:
            return None
        else:
            return post_interactions[0]

    def get_action(self):
        """TODO conflate decision and action"""
        return int(self.get_decision())
        
    def get_decision(self):
        """Return the flatten sequence of intermediary primitive interactions terminated with the final decision"""
        if self.get_post_interaction() is None:
            # Composite event: bottom of the recursion
            return self.get_pre_interaction().get_decision()
            # return self.get_pre_interaction().get_decision()
        else:
            sequence = self.sequence() 
            return f"{self.sequence()[:-1]}{sequence[-1].get_decision()}"
            # return f"{self.get_pre_interaction().sequence()}{self.get_post_interaction().get_decision()}"

    def get_actions(self):
        """Return the flat sequence of the decisions of this interaction as a string"""
        if self.get_post_interaction() is None:
            # Composite event: bottom of the recursion 
            return self.get_decision()
        else: 
            return f"{self.get_pre_interaction().get_actions()}{self.get_post_interaction().get_actions()}"
    
    def get_valence(self):
        """Return the valence of the pre_interaction plus the valence of the post_interaction"""
        return sum(i.get_valence() for i in self._interactions.values())  # Not sure whether we should sum the valences of composite events
        # return self.get_pre_interaction().get_valence() + self.get_post_interaction().get_valence()

    def reinforce(self):
        """Increment the composite interaction's weight"""
        self.weight += 1

    def key(self):
        """ Return a string identifier that preserves the hierarchy"""
        # Works as a unique identifier if the coordinates are sorted
        if self.get_post_interaction() is None:
            return "[" + ",".join(i.key() for i in self._interactions.values()) + "]"  # Composite event
        else:
            return "\u27E8" + ",".join(i.key() for i in self._interactions.values()) + "\u27E9" 

    def __str__(self):
        """ Print the interaction in the Newick tree format (pre_interaction, post_interaction: valence) """
        return f"{self.key()}w{self.weight}" 
        # return f"({",".join(i.key() for i in self._interactions.values())}:{self.weight})" 
        # return f"({self.get_pre_interaction()}, {self.get_post_interaction()}: {self.weight})"

    def __repr__(self):
        return self.__str__()
        
    def __eq__(self, other):
        """ CompositeInteractions are equal if they have the same dictionaries of interactions (key and values)"""
        if isinstance(other, CompositeInteraction):
            # Two dicts are equal if and only if they have the same set of keys and each key maps to an equal value
            return self._interactions == other._interactions
            # return (self.get_pre_interaction() == other.get_pre_interaction()) and (self.get_post_interaction() == other.get_post_interaction())
        elif isinstance(other, Interaction):
            return False
        else:
            raise TypeError("Must compare with an interaction or a CompositeInteraction") 
    
    def get_length(self):
        """Return the number of primitive interactions in this composite interaction"""
        if self.get_post_interaction() is None:
            return 1
        else:
            return self.get_pre_interaction().get_length() + self.get_post_interaction().get_length()
        # return sum(i.get_length() for i in self._interactions.values())

    def increment(self, interaction, interactions):
        """Increment the step of the appropriate sub-interaction. 
           Add or reinforce the neacted composite interaction at this level of the hiearchy.
           Return the enacted interaction if it is over, or None if it is ongoing."""
        # First step 
        if self._step == 1:
            interaction = self.get_pre_interaction().increment(interaction, interactions)
            # Ongoing pre-interaction. Return None
            if interaction is None:
                return None
            # Pre-interaction succeeded. Increment the step and return None
            elif interaction == self.get_pre_interaction():
                self._step = 2
                return None
            # Pre-interaction failed. Reset the step and return the enacted interaction
            else:
                self._step = 1
                return interaction
        # Second step
        else:
            interaction = self.get_post_interaction().increment(interaction, interactions)
            # Ongoing post-interaction. Return None
            if interaction is None:
                return None
            # Post-interaction succeeded. Reset the step and return this interaction
            elif interaction == self.get_post_interaction():
                self._step = 1
                return self
            # Post-interaction failed. Reset the step and return the enacted interaction
            else:
                self._step = 1
                # composite_interaction = CompositeInteraction(self.get_pre_interaction(), interaction)
                composite_interaction = CompositeInteraction({Coordinate(0, PRE):self.get_pre_interaction(), Coordinate(0, POST):interaction})
                if composite_interaction.key() not in interactions:
                    # Add the enacted composite interaction to memory
                    interactions[composite_interaction.key()] = composite_interaction
                    # print(f"Learning {composite_interaction}")
                    return composite_interaction
                else:
                    # Reinforce the existing composite interaction and return it
                    interactions[composite_interaction.key()].reinforce()
                    # print(f"Reinforcing {interactions[composite_interaction.key()]}")
                    return interactions[composite_interaction.key()]

    def current(self):
        """Return the current intended primitive interaction"""
        # Step 1: the current primitive interaction of the pre-interaction
        if self._step == 1:
            if self.get_post_interaction() is None:
                # If composite event, end of the recusion
                return self
            else:
                return self.get_pre_interaction().current()
        # Step 2: The current primitive interaction of the post-interaction
        else:
            return self.get_post_interaction().current()

    def sequence(self):
        """Return the flat list of primitive interactions of this composite interaction"""
        if self.get_post_interaction() is None:
            # Composite Event: bottom of the recursion
            return list(self._interactions.values())
            # return [self.get_pre_interaction()]
        else:
            return self.get_pre_interaction().sequence() + self.get_post_interaction().sequence()  # Append the two lists
        # return f"{self.get_pre_interaction().sequence()}{self.get_post_interaction().sequence()}"

    def get_post_interactions(self):
        """Return the list of the hierarchy of the sub post_interactions"""
        if self.get_post_interaction() is None:
            return []
        return [self.get_post_interaction()] + self.get_post_interaction().get_post_interactions()

In [123]:
class Interaction:
    """An interaction is a tuple (action, outcome) with a valence"""
    def __init__(self, _action, _outcome, _valence):
        self._action = _action
        self._outcome = _outcome
        self._valence = _valence
        self.weight = 10
        
    def get_action(self):
        """Return the action"""
        return self._action

    def get_actions(self):
        """Return the action as a string for compatibilty with CompositeInteraction"""
        return str(self._action)

    def get_decision(self):
        """Return the decision key"""
        return f"{self._action}"
        # return f"a{self._action}"

    def get_outcome(self):
        """Return the action"""
        return self._outcome

    def get_valence(self):
        """Return the action"""
        return self._valence

    def key(self):
        """ The key to find this interaction in the dictinary is the string '<action><outcome>'. """
        return f"{self._action}{self._outcome}"

    def __str__(self):
        """ Print interaction in the form '<action><outcome:<valence>' for debug."""
        return f"{self._action}{self._outcome}:{self._valence}"

    def __eq__(self, other):
        """ Primitive interactions are equal if they have the same key """
        if isinstance(other, Interaction):
            return self.key() == other.key()
        elif isinstance(other, CompositeInteraction):
            return False
        else:
            raise TypeError("Must compare with an interaction or a CompositeInteraction") 

    def get_length(self):
        """The length of the sequence of this interaction"""
        return 1

    def increment(self, interaction, interactions):
        """Return the enacted interaction for compatibility with composite interactions"""
        return interaction

    def current(self):
        """Return itself for compatibility with composite interactions"""
        return self

    def sequence(self):
        """Return a list containing this interaction only """
        return [self]

    def get_post_interactions(self):
        """Return the empty list for compatibility with composite interactions"""
        return []

    def __repr__(self):
        """Return the string representation of the interaction for tracing"""
        return self.__str__()

## Class PositionIndex

Définit la position d'un EIV par son index spacial et son index temporel

In [124]:
class PositionIndex():
    """A spatio-temporal position for an EIV"""
    def __init__(self, spatial, temporal):
        # Spatial and temporal coordinates of an EIV
        self._spatial = spatial  # 0: Front, 1: Left, 2: Right
        self._temporal = temporal
    def get_spatial(self):
        """Return the spatio temporal index without the last item"""
        return self._spatial
    def get_temporal(self):
        """Return temporal part"""
        return self._temporal
    def __hash__(self):
        """Return a unique integer to use as a key in a dictionary"""
        assert self._spatial < 10 , "The spatial index must be lower than 10"
        return self._spatial + 10 * self._temporal 
    def __eq__(self, other):
        """PositionIndex are equal if they have the same hash, to use as a key in a dictionary"""
        return hash(self) == hash(other)
    def __repr__(self):
        """Return a string representation for tracing"""
        return f"{repr(self._spatial)}{repr(self._temporal)}"
    def __lt__(self, other):
        """A PositionIndex is lower that another if its hash is lower. Used to sort the sub interactions"""
        return hash(this) < hash(other)

# L'agent

Nous repartons de l'Agent10 avec les modifications:
* Lorsque plusieurs EIV primitifs sont énacatés simultanément, ils sont regroupés dans un un événement composite.
* La méthode `learn_composite_interaction()` prend comme argument le pattern spatio-temporel des interactions qui composeront cette intéraction composite.
* Les décisions primitives sont ajoutées dans `decision_df` au lieu de passer par `proposed_df`. 

In [147]:
import pandas as pd
trace = True

class Agent:
    def __init__(self, _interactions, simulator):
        """ Initialize our agent """
        self._interactions = {interaction.key(): interaction for interaction in _interactions}
        self._primitive_intended_interaction = self._interactions[f"{FORWARD}0"]
        self._intended_interaction = None
        self._simulator = simulator
        # The context
        self._penultimate_interaction = None
        self._previous_interaction = None
        self._last_interaction = None
        self._penultimate_composite_interaction = None
        self._previous_composite_interaction = None
        self._last_composite_interaction = None
        self._permutation_index = 0
        self._body_model = body_model
        # self._simulated_outcomes = []
        self.proposed_df = None
        self.decision_df = None
        self.clear = True # Used to clear the display after the enacted interaction
        default_decisions = [interaction.get_decision() for interaction in _interactions if interaction.get_outcome() == 0]
        self._default_decision_df = pd.DataFrame({'decision': default_decisions, 'proclivity': 0})

        self._expected_content = {}
        self._expected_pose = None

    def action(self, _outcome):
        """Implement the agent's policy"""

        # The list of enacted primitive interactions
        primitive_enacted_interactions = [self._interactions[f"{self._primitive_intended_interaction.get_action()}{o}"] for o in _outcome]
        # The composite interaction containing one or several simultaneously enacted EIVs
        pattern = {PositionIndex(rank, PRE):interaction for rank, interaction in enumerate(primitive_enacted_interactions)}
        simultaneous_enacted_interaction = self.learn_composite_interaction(pattern) 

        # Trace the previous cycle
        if trace:
            print(
            f"Enacted primitive localized interactions: {repr(simultaneous_enacted_interaction)}") # ,\n" 
            # f"Sequential predicted outcome: {self._primitive_intended_interaction.get_outcome()}, "
            # f"Simulator predicted outcomes: {self._simulated_outcomes}, "
            # f"Prediction_correct: {self._primitive_intended_interaction.get_outcome() == _outcome[0]}, ")

        # Synchronize the simulator
        experienced_content, experienced_pose = self._simulator.simulate(simultaneous_enacted_interaction, False)
        # Update the GEITH or resolve the inconsistencies
        self.resolve(experienced_content, experienced_pose)

        # Follow up the enaction
        if self._intended_interaction is None: # First interaction cycle
            enacted_interaction = simultaneous_enacted_interaction
        else:
            enacted_interaction = self._intended_interaction.increment(simultaneous_enacted_interaction, self._interactions)

        # If the intended interaction is over (completely enacted or aborted)
        if enacted_interaction is None:
            self.clear = False
        else:
            self.clear = True
            # Memorize the context
            self._penultimate_composite_interaction = self._previous_composite_interaction
            self._previous_composite_interaction = self._last_composite_interaction
            self._penultimate_interaction = self._previous_interaction
            self._previous_interaction = self._last_interaction
            self._last_interaction = enacted_interaction
            # Call the learning mechanism
            self.learn(enacted_interaction)
            # Create the proposed dataframe
            self.create_proposed_df()
            self.aggregate_propositions()
            # The decision is the first row in the decision_df
            decision_key = self.decision_df.loc[0, 'decision']
            if pd.isna(self.decision_df.loc[0, "intention"]):
                # Pick the first interaction that has this decision
                self._intended_interaction = next((i for i in self._interactions.values() if i.get_decision() == decision_key), None)
            else:
                self._intended_interaction = self._interactions[self.decision_df.loc[0, "intention"]]
            if trace:
                print("Intention:", self._intended_interaction.key())

        # Return the next primitive decision
        self._primitive_intended_interaction = self._intended_interaction.current()       
        # Simulate the intended primitive interaction to produce an expected content and pose
        if isinstance(self._primitive_intended_interaction, CompositeInteraction):
            self._expected_content, self._expected_pose = self._simulator.simulate(self._primitive_intended_interaction, True)
        else:
            self._expected_content, self._expected_pose = ({}, None)            
        return self._primitive_intended_interaction.get_action()
        
    def learn(self, enacted_interaction):
        """Learn the composite interactions from a pattern of enacted interactions"""
        # First level of composite interactions
        if self._previous_interaction is not None:
            pattern = {PositionIndex(0, PRE):self._previous_interaction, PositionIndex(0, POST):enacted_interaction}
            self._last_composite_interaction = self.learn_composite_interaction(pattern)
        # Second level of composite interactions
        if self._previous_composite_interaction is not None:
            pattern = {PositionIndex(0, PRE):self._previous_composite_interaction, PositionIndex(0, POST):enacted_interaction}
            self.learn_composite_interaction(pattern)
        if self._penultimate_interaction is not None:
            pattern = {PositionIndex(0, PRE):self._penultimate_interaction, PositionIndex(0, POST):self._last_composite_interaction}
            self.learn_composite_interaction(pattern)
        # Higher level composite interaction made of two composite interactions
        if self._penultimate_composite_interaction is not None and self._last_composite_interaction is not None:
            pattern = {PositionIndex(0, PRE):self._penultimate_composite_interaction, PositionIndex(0, POST):self._last_composite_interaction}
            self.learn_composite_interaction(pattern)

    def learn_composite_interaction(self, pattern):
        """Create or reinforce the composite interaction made of this pattern. Return the new or existing composite interaction."""
        composite_interaction = CompositeInteraction(pattern)
        if composite_interaction.key() not in self._interactions:
            # Add the composite interaction to memory
            self._interactions[composite_interaction.key()] = composite_interaction
            if trace:
                print(f"Learning {composite_interaction}")
            return composite_interaction
        else:
            # Reinforce the existing composite interaction and return it
            self._interactions[composite_interaction.key()].reinforce()
            if trace:
                print(f"Reinforcing {self._interactions[composite_interaction.key()]}")
            return self._interactions[composite_interaction.key()]

    def create_proposed_df(self):
        """Create the proposed dataframe from the activated interactions"""
        # The list of activated interactions that match the current context
        activated_interactions = [i for i in self._interactions.values() if i.get_length() > 1 
                                  and i.get_pre_interaction() in self._last_composite_interaction.get_post_interactions()]
        self.proposed_df = pd.DataFrame({
            'activated': [i.key() for i in activated_interactions],
            'weight': [i.weight for i in activated_interactions],
            'actions': [i.get_post_interaction().get_actions() for i in activated_interactions],
            'intention': [i.get_post_interaction().key() for i in activated_interactions],
            'valence': [i.get_post_interaction().get_valence() for i in activated_interactions],
            'decision': [i.get_post_interaction().get_decision() for i in activated_interactions],
            'length': [i.get_post_interaction().get_length() for i in activated_interactions]})

        # Calculate the proclivity of each proposition
        self.proposed_df['proclivity'] = self.proposed_df['weight'] * self.proposed_df['valence']

    def aggregate_propositions(self):
        """Aggregate the proclivity"""
        # Aggregate the proclivity for each decision
        decision_df = self.proposed_df.groupby('decision').agg({'proclivity': 'sum', 'actions': 'first', 'length': 'first', 
                                                                'intention': 'first'}).reset_index()
        # For each proposed composite decision 
        for index, proposed in decision_df[decision_df['length'] > 1].iterrows():
            # print(f"Index {index}, actions {proposition['actions']}, intention {proposition['intention']}")
            # Find shorter decisions that start with the same sequence 
            for _, shorter in self.proposed_df[self.proposed_df.apply(lambda row: proposed['actions'].startswith(row['actions']) 
                                                                      and row['length'] < proposed['length'], axis=1)].iterrows():
                # Add the proclivity of the shorter decisions
                decision_df.loc[index, 'proclivity'] += shorter['proclivity']
                # print(f"Decision {proposed['decision']} recieves {shorter['proclivity']} from shorter {shorter['intention']}")

        # Concatenate with default decisions (the intention will come from decision_df because it is the first non NaN)
        self.decision_df = pd.concat([self._default_decision_df, decision_df]).groupby("decision", as_index=False).agg({"proclivity": "sum", 
                                                                                                                      "intention":"first"})
        # Sort by descending proclivity
        self.decision_df = self.decision_df.sort_values(by=['proclivity', 'decision'], ascending=[False, True]).reset_index(drop=True)

    def simulate(self):
        """Simulate the intended interaction in a cloned simulator and display it. """
        current_simulation = self._simulator.clone()
        for primitive_interaction in self._intended_interaction.sequence():
            primitive_decision = primitive_interaction.get_action()
            simulated_outcomes = current_simulation.predict(primitive_decision)
            consistent = simulated_outcomes[0] in [primitive_interaction.get_outcome(), UNKNOWN]
            print("Simulate decision:", primitive_decision, "anticipated outcomes:", simulated_outcomes, "consistent:", consistent)
            current_simulation.marker_color = simulator_agent_color if consistent else prediction_error_color
            e.display(current_simulation)

    def resolve(self, experienced_content, experienced_pose):
        """Resolve the inconsistencies: just reset and permutate the geith if an inconsistency is found"""
        # For all keys in the expected_content, check that their values are the same in the experienced_content
        consistent = True
        for simultator_position, content in self._expected_content.items():
            if experienced_content[simultator_position] != content:
                self._simulator.display_grid[simultator_position] = PREDICTION_ERROR
                consistent = False
        if self._expected_pose is not None and self._expected_pose != experienced_pose:
            self._simulator.marker_color = prediction_error_color
            consistent = False
        if consistent:
            # Update the GEITH with the experienced content and pose
            self._simulator.update(experienced_content, experienced_pose)
        else:
            # Briefly display the expectation errors
            e.display(self._simulator)
            # Reset the geith
            self._simulator.reset()
            # Try the next permutation
            self._permutation_index = (self._permutation_index + 1) % len(model_permutations)
            print("-- Adjusting the GEITH to index", self._permutation_index, "permutation", model_permutations[self._permutation_index])
            body_model[(FEEL, 0)] = model_permutations[self._permutation_index][0]
            body_model[(FEEL, 1)] = model_permutations[self._permutation_index][1]
            body_model[(FEEL, 2)] = model_permutations[self._permutation_index][2]
            # action_model[f"{TURN_LEFT}0"] = [model_permutations[self._permutation_index][3]]
            # action_model[f"{TURN_RIGHT}0"] = [model_permutations[self._permutation_index][4]]

# Le simulateur

Le simulateur de Agent10 est inchangé.

In [164]:
KILL = 0
SPAWN = 1
TRANSLATE = 2
ROTATE_COUNTERCLOCKWISE = 3
ROTATE_CLOCKWISE = 4

SIM_SIZE = np.array([SIM_HIGHT, SIM_WIDTH])

class Simulator():
    def __init__(self, pose, grid):
        self.init_grid = np.array(grid)
        self.simulation_grid = np.array(grid)
        self.display_grid = np.array(grid)
        self.pose = pose.clone()  # Must clone the pose
        self.marker_size = agent_size
        self.marker_color = simulator_agent_color
        self.expected_contents = {}
        self.expected_pose = None
        self.experienced_contents = {}
        self.experienced_pose = None

    def simulate(self, simultaneous_enacted_interaction, expected):
        "Simulate the interactions. Return the content and the pose that are expected or experienced."""
        content = {}  # The contents expected or experienced. Keys are typles representing the position in the simulator
        pose = self.pose.clone()  # The pose expected or experienced
        for position_index, interaction in simultaneous_enacted_interaction.simultaneous_interactions(PRE).items():
            # Get the simulator position from the body_model
            spatial_body_index = body_model.get((interaction.get_action(), position_index.get_spatial()), 0)
            # position_index = body_model.get((interaction.get_action(), coordinate.get_spatial()), 0)
            simulator_position = [self.pose.front(), self.pose.left(), self.pose.right()][spatial_body_index]
            # simulator_position = [self.pose.front(), self.pose.left(), self.pose.right()][coordinate.get_spatial()]
            # Simulate the interaction
            if interaction.get_action() == FORWARD: 
                if expected: 
                    if self.simulation_grid[simulator_position] == WALL:
                        content[simulator_position] = WALL
                    if self.simulation_grid[simulator_position] == EMPTY:
                        content[simulator_position] = EMPTY
                        pose.forward()
                    else: 
                        # If UNKNWON: do not expect a content but expect to move forward (generate an inconsistency if results in bump)
                        pose.forward()
                else:
                    content[simulator_position] = interaction.get_outcome()
                    if interaction.get_outcome() == EMPTY:
                        pose.forward()
                    # Do not move fowards if outcome WALL
            elif interaction.get_action() == TURN_LEFT:
                pose.turn_left()
            elif interaction.get_action() == TURN_RIGHT:
                pose.turn_right()
            elif interaction.get_action() == FEEL:
                if expected: 
                    if self.simulation_grid[simulator_position] != UNKNOWN: 
                        content[simulator_position] = int(self.simulation_grid[simulator_position])
                    # Do not expect a content if UNKNOWN
                else:
                    content[simulator_position] = interaction.get_outcome()
        if expected: 
            print("Expected content at pose", repr(self.pose), "is:", repr(content), ", next expected pose:", repr(pose))
        else:
            print("Experienced content at pose", repr(self.pose), "is:", repr(content), ", next experienced pose:", repr(pose))
        return content, pose

    def update(self, experienced_content, experienced_pose):
        """Update the GEITH according to the experienced content and pose"""
        for simulator_position, content in experienced_content.items():
            self.simulation_grid[simulator_position] = content
            if trace:
                print(f"Update content {content} at {simulator_position}")
        self.pose = experienced_pose
        self.display_grid[:SIM_HIGHT, :SIM_WIDTH] = self.simulation_grid
    
    def clone(self):
        """Return a copy of the simulator in its current state"""
        return Simulator(self.pose, self.simulation_grid)

    def reset(self):
        """Clear the simulator and move the agent to the origin"""
        self.simulation_grid[:, :] = self.init_grid
        self.display_grid[:, :] = self.init_grid
        self.marker_color = simulator_agent_color
        self.pose = simulator_pose.clone()

# On initialise les modèles de GEITH

Les modèles du GEITH de l'Agent10 sont inchangés.

In [310]:
from itertools import permutations, product

# Le body_model maps (decicion, outcome rank) to spatial position in the simulator
body_model = {(FEEL, 0): 2,  # Front
              (FEEL, 1): 1,  # Left 
              (FEEL, 2): 0,  # Right
              (FORWARD, 0): 0 # (-1, 0) # Bump
}
# action_model maps the primitive interactions to a list of actions to perfom in the simulator
action_model = {
    f"{FEEL}0": [KILL], 
    f"{FEEL}1": [SPAWN], 
    f"{FORWARD}0": [KILL, TRANSLATE],
    f"{FORWARD}1": [SPAWN],
    f"{TURN_LEFT}0": [ROTATE_CLOCKWISE],
    f"{TURN_RIGHT}0": [ROTATE_COUNTERCLOCKWISE]
}
# object_model maps the decisions to the outcome produced by this object (there will be one object_model per type of object)
object_model = {FEEL:WALL,
                FORWARD: WALL}

# Possible permutations
model_permutations = [a + b for a, b in product(permutations([1, 0, 2]), permutations([ROTATE_CLOCKWISE, ROTATE_COUNTERCLOCKWISE]))]
model_permutations = [a + b for a, b in product(permutations([2, 1, 0]), permutations([ROTATE_CLOCKWISE, ROTATE_COUNTERCLOCKWISE]))] #  inverted GEITH
model_permutations = tuple(permutations([2, 1, 0]))
model_permutations

((2, 1, 0), (2, 0, 1), (1, 2, 0), (1, 0, 2), (0, 2, 1), (0, 1, 2))

# Testons l'agent

On donne une valence de $-5$ pour `TURN` pour qu'elle soit plus négative que les trois `FEEL`. 

In [311]:
# Empty grid with sparse objects
grid = [[WALL , WALL , WALL , WALL , WALL ],
        [WALL , EMPTY, EMPTY, EMPTY, WALL ],
        [WALL , EMPTY, WALL , EMPTY, WALL ],
        [WALL , EMPTY, EMPTY, EMPTY, WALL ],
        [WALL , WALL , WALL , WALL , WALL ]]

In [312]:
# Instanciate the environment
e = SmallLoop([1, 1], 0, grid)

# Instanciate the agent 
interactions = [
    Interaction(FORWARD,0,5),
    Interaction(FORWARD,1,-10),
    Interaction(TURN_LEFT,0,-5),
    # Interaction(TURN_LEFT,1,-3),
    Interaction(TURN_RIGHT,0,-5),
    # Interaction(TURN_RIGHT,1,-3),
    Interaction(FEEL,0,-1),
    Interaction(FEEL,1,-1) # ,
    # Interaction(FEEL_LEFT,0,-1),
    # Interaction(FEEL_LEFT,1,-1),
    # Interaction(FEEL_RIGHT,0,-1),
    #Interaction(FEEL_RIGHT,1,-1)
]
# Intialize the simulator to UNKNOWN grid
simulator_pose = Pose([(SIM_WIDTH - 1) // 2, (SIM_HIGHT - 1) // 2], UP)  # Agent at the center of the simulator assuming size is even
simulator_grid = np.full((SIM_HIGHT, SIM_WIDTH), UNKNOWN, dtype=int)
# simulator_grid[tuple(simulator_pose.position)] = EMPTY  # Empty cell at the initial position of the agent
simulator = Simulator(simulator_pose, simulator_grid)
a = Agent(interactions, simulator)

# Initialize the interaction loop
step = 0
outcome = [0]

# Display
out = Output()
e.display(simulator)
display(out)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 640x480 with 1 Axes>', 'i…

In [328]:
trace = True
step += 1
print("Step:", step)
action = a.action(outcome)
outcome = e.outcome(action)
e.display(simulator)
a.decision_df

Step: 16
Reinforcing [21]w4
Enacted primitive localized interactions: [21]w4
Experienced content at pose (2,1)v is: {(3, 1): 1} , next experienced pose: (2,1)v
-- Adjusting the GEITH to index 2 permutation (1, 2, 0)
Learning ⟨[40],[21]⟩w1
Learning ⟨⟨[20],[40]⟩,[21]⟩w1
Learning ⟨[20],⟨[40],[21]⟩⟩w1
Learning ⟨⟨[40],[20]⟩,⟨[40],[21]⟩⟩w1
Intention: 50


,decision,proclivity,intention
0,5,0,NaN
1,1,-3,"[11,10,11]"
2,4,-5,[40]
3,[40:-5]2,-5,"⟨[40],[20]⟩"
4,2,-10,[21]
5,"[11:-1, 10:-1, 11:-1]4",-11,"⟨[11,10,11],[40]⟩"
6,[21:-10]4,-25,"⟨[21],[40]⟩"


## Run the experiment in a loop 

In [ ]:
for step in range(100):
    
    if trace:
        print(f"Step: {step}")
    # step += 1
    action = a.action(outcome)
    outcome = e.outcome(action)
    e.display(simulator)

## Execute step by step for testing

Note that the simulator is synchronized after the next cycle when the outcome has been recieved. 

# Questions

* Est-ce qu'il ne faudrait pas que le mécanisme de schéma interagisse avec le GEITH en priorité par rapport à l'environnement ? 
    * Les décisions du mécanisme de schéma seraient donc des actions sur le GEITH: TRANSLATE, ROTATE, SPAWN, KILL
    * L'`action_model` enregistrerait les correspondance de ces actions vers les interactions avec l'environnement
    * Citer The Natural Origin of Content de Daniel Hutto https://link.springer.com/article/10.1007/s11406-015-9644-0
        * Passage d'un _intentional patients_ à un _intentional agents_ 
* Comment unifier les méthodes `synchronize()` et `predict()`du simulateur ?
    * `synchronize()` prend une liste d'interactions localisées
    * Elle renvoie un dictionnaire de `expected_contents` pour les cellules qui ne sont pas unknown
        * Ce dictionnaire pourra servir à générer les prediciton errors 
    * Elle renvoie le dictionnaire des `afforded_contents` affordés par les interactions énactées
    * Les `prediction errors` sont les `afforded_contents` contradictoires avec les `expected_contents` 
        * Leur traitement se fait par adjudication selon le principe de _epistemic resolve_ (Khellar Crowford 2026)
    * Elle prend comme argument une interaction simultanée